# Feature 7: Personality Archetype Detection

## Objective

Assign exactly one personality archetype to every participant based on their messaging behaviour.

This feature combines the outputs of previous features and applies quantitative rules to identify unique behavioural patterns for each participant.

## Personality Archetypes

- THE SPAMMER
- THE GROUP MOM
- THE NIGHT OWL
- THE STORYTELLER
- THE DRAMA QUEEN
- THE GHOST
- THE COMEDIAN
- THE QUESTION MASTER

## Input Files

- parsed_chat.pkl
- group_overview.pkl
- activity_summary.pkl
- activity_heatmap.pkl
- word_frequency.pkl
- response_analysis.pkl

## Output

- Archetype score for every participant
- Final personality archetype
- Validation
- personality_archetypes.pkl

In [1]:
import pickle

In [2]:
DATA_PATH = "../Data/"

with open(DATA_PATH + "parsed_chat.pkl", "rb") as file:
    chat_data = pickle.load(file)

with open(DATA_PATH + "group_overview.pkl", "rb") as file:
    group_overview = pickle.load(file)

with open(DATA_PATH + "activity_summary.pkl", "rb") as file:
    activity_summary = pickle.load(file)

with open(DATA_PATH + "activity_heatmap.pkl", "rb") as file:
    activity_heatmap = pickle.load(file)

with open(DATA_PATH + "word_frequency.pkl", "rb") as file:
    word_frequency = pickle.load(file)

with open(DATA_PATH + "response_analysis.pkl", "rb") as file:
    response_analysis = pickle.load(file)

In [3]:
print("=" * 70)
print("FILES LOADED")
print("=" * 70)

print(f"Chat Data          : {type(chat_data).__name__}")
print(f"Group Overview     : {type(group_overview).__name__}")
print(f"Activity Summary   : {type(activity_summary).__name__}")
print(f"Activity Heatmap   : {type(activity_heatmap).__name__}")
print(f"Word Frequency     : {type(word_frequency).__name__}")
print(f"Response Analysis  : {type(response_analysis).__name__}")

FILES LOADED
Chat Data          : list
Group Overview     : dict
Activity Summary   : dict
Activity Heatmap   : dict
Word Frequency     : dict
Response Analysis  : dict


In [4]:
# TEXT PROCESSING HELPERS AND KEYWORD SETS

def extracting_words(text):
    words = []

    for word in text.lower().split():
        cleaned = ""

        for character in word:
            if character.isalpha():
                cleaned += character

        if cleaned:
            words.append(cleaned)

    return words


def counting_keywords(words, keyword_set):
    count = 0

    for word in words:
        if word in keyword_set:
            count += 1

    return count


CARING_KEYWORDS = {
    "care", "safe", "careful", "please",
    "eat", "khao", "kheye", "khabi", "khawa", "khabar",
    "sleep", "ghum", "ghumiye", "ghumabi", "ghumiyeo",
    "water", "jal", "pani", "drink",
    "medicine", "med", "tablet", "oshudh",
    "remember", "mone", "bhul", "forgot", "reminder",
    "rest", "valo", "bhalo", "thik", "thikache",
    "okay", "ok",
    "sabdhane", "reach", "reached", "home", "bari"
}

DRAMA_KEYWORDS = {
    "ki", "keno", "kiii", "kiiii",
    "arre", "arey", "are", "oye",
    "bro", "broo", "brooo",
    "bhai", "bhaiii",
    "wtf", "omg", "uff", "ufff",
    "nah", "nahh", "no", "nooo",
    "please", "pls", "plzz", "plzzz",
    "seriously", "really", "sotti", "sotty",
    "wow", "wah", "unbelievable",
    "impossible", "can't", "cant"
}

QUESTION_KEYWORDS = {
    "what",
    "why",
    "when",
    "where",
    "who",
    "whom",
    "whose",
    "which",
    "how",
    "is",
    "are",
    "can",
    "could",
    "should",
    "will",
    "would",
    "did",
    "do",
    "does",
    "any",
    "ki",
    "keno",
    "kobe",
    "kokhon",
    "kothay",
    "kotay",
    "ke",
    "kar",
    "kake",
    "kotota",
    "kotobar"
}

COMEDIAN_KEYWORDS = {
    "lol",
    "lmao",
    "lmfao",
    "rofl",
    "haha",
    "hahaha",
    "hehe",
    "hehehe",
    "xd",
    "xdd"
}

COMEDIAN_EMOJIS = {
    "😂",
    "🤣",
    "😆",
    "😹"
}

In [5]:
def valid_messages(chat_data):
    filtered_messages = []

    for message in chat_data:
        sender = message["sender"]

        if sender in IGNORE_PARTICIPANTS:
            continue

        if any(pattern in sender for pattern in IGNORE_PATTERNS):
            continue

        filtered_messages.append(message)

    return filtered_messages

In [6]:
IGNORE_PARTICIPANTS = {
    "Meta AI"
}

IGNORE_PATTERNS = [
    "changed the group name",
    "changed this group's icon",
    "created group",
    "added",
    "removed",
    "joined using this group's invite link",
    "left"
]

def calculating_participants(chat_data):
    participants = []

    for message in chat_data:
        sender = message["sender"]

        if sender in IGNORE_PARTICIPANTS:
            continue

        if any(pattern in sender for pattern in IGNORE_PATTERNS):
            continue

        if sender not in participants:
            participants.append(sender)

    return participants


def calculating_messages_per_person(chat_data):
    messages_per_person = {}

    for message in chat_data:
        sender = message["sender"]

        if sender in IGNORE_PARTICIPANTS:
            continue

        if any(pattern in sender for pattern in IGNORE_PATTERNS):
            continue

        if sender not in messages_per_person:
            messages_per_person[sender] = []

        messages_per_person[sender].append(message)

    return messages_per_person


def calculating_words(text):
    return len(extracting_words(text))


def calculating_percentage(part, whole):
    if whole == 0:
        return 0

    return (part / whole) * 100

In [7]:
participants = calculating_participants(chat_data)
messages_per_person = calculating_messages_per_person(chat_data)
valid_chat_data = valid_messages(chat_data)

print("=" * 70)
print("HELPER FUNCTIONS")
print("=" * 70)

print(f"Participants : {len(participants)}")
print(f"Grouped Chats : {len(messages_per_person)}")
print(f"Valid Messages : {len(valid_chat_data)}")

HELPER FUNCTIONS
Participants : 75
Grouped Chats : 75
Valid Messages : 78047


In [8]:
archetype_scores = {
    "THE SPAMMER": {},
    "THE GROUP MOM": {},
    "THE NIGHT OWL": {},
    "THE STORYTELLER": {},
    "THE DRAMA QUEEN": {},
    "THE GHOST": {},
    "THE COMEDIAN": {},
    "THE QUESTION MASTER": {}
}

print("=" * 70)
print("ARCHETYPE SCORE MATRIX")
print("=" * 70)

for archetype in archetype_scores:
    print(archetype)

ARCHETYPE SCORE MATRIX
THE SPAMMER
THE GROUP MOM
THE NIGHT OWL
THE STORYTELLER
THE DRAMA QUEEN
THE GHOST
THE COMEDIAN
THE QUESTION MASTER


In [9]:
# THE SPAMMER

def calculating_spammer_scores(valid_chat_data):
    participant_bursts = {}
    current_sender = chat_data[0]["sender"]
    current_burst = 1

    for message in chat_data[1:]:
        sender = message["sender"]

        if sender == current_sender:
            current_burst += 1
        else:
            participant_bursts.setdefault(current_sender, [])
            participant_bursts[current_sender].append(current_burst)

            current_sender = sender
            current_burst = 1

    participant_bursts.setdefault(current_sender, [])
    participant_bursts[current_sender].append(current_burst)

    for participant in participants:
        bursts = participant_bursts.get(participant, [])

        if not bursts:
            archetype_scores["THE SPAMMER"][participant] = 0
        else:
            archetype_scores["THE SPAMMER"][participant] = sum(bursts) / len(bursts)

In [10]:
calculating_spammer_scores(valid_chat_data)

print("=" * 70)
print("THE SPAMMER")
print("=" * 70)

top_spammers = sorted(
    archetype_scores["THE SPAMMER"].items(),
    key=lambda item: item[1],
    reverse=True
)

print(f"{'Rank':<6}{'Participant':<35}{'Average Burst'}")
print("-" * 70)

for rank, (participant, score) in enumerate(top_spammers[:10], start=1):
    print(f"{rank:<6}{participant:<35}{score:.2f}")

THE SPAMMER
Rank  Participant                        Average Burst
----------------------------------------------------------------------
1     Anshley CSE A                      1.86
2     Shambaditya Sarkar CSE             1.85
3     +91 93303 29288                    1.72
4     Mohok CSE                          1.69
5     +91 82405 93621                    1.68
6     Sagnik CSE                         1.62
7     Souryadip Cse                      1.62
8     Debayan Bhowmik CSE                1.60
9     Saptarshi CSE                      1.60
10    +91 70017 73199                    1.57


In [11]:
# THE GROUP MOM

def calculating_group_mom_scores(messages_per_person):
    for participant in participants:
        caring_count = 0

        for message in messages_per_person[participant]:
            words = extracting_words(message["text"])
            caring_count += counting_keywords(words, CARING_KEYWORDS)

        archetype_scores["THE GROUP MOM"][participant] = caring_count

In [12]:
calculating_group_mom_scores(messages_per_person)

print("=" * 70)
print("THE GROUP MOM")
print("=" * 70)

top_group_moms = sorted(
    archetype_scores["THE GROUP MOM"].items(),
    key=lambda item: item[1],
    reverse=True
)

print(f"{'Rank':<6}{'Participant':<35}{'Caring Count'}")
print("-" * 70)

for rank, (participant, score) in enumerate(top_group_moms[:10], start=1):
    print(f"{rank:<6}{participant:<35}{score}")

THE GROUP MOM
Rank  Participant                        Caring Count
----------------------------------------------------------------------
1     Kaninika CSE                       564
2     Soumili Ghosh CSE                  285
3     Sagnik CSE                         272
4     Shambaditya Sarkar CSE             259
5     Shubham CSE Lambu                  203
6     Shubham Biswas                     196
7     Mohok CSE                          181
8     Saptarshi CSE                      133
9     Anshley CSE A                      131
10    Souryadip Cse                      115


In [13]:
MIN_MESSAGES = 20

def calculating_night_owl_scores(valid_chat_data):
    total_messages = {participant: 0 for participant in participants}
    night_messages = {participant: 0 for participant in participants}

    for message in valid_chat_data:
        participant = message["sender"]
        hour = message["timestamp"].hour

        total_messages[participant] += 1

        if 0 <= hour <= 4:
            night_messages[participant] += 1

    for participant in participants:
        total = total_messages[participant]
        night = night_messages[participant]

        if total < MIN_MESSAGES:
            archetype_scores["THE NIGHT OWL"][participant] = 0
        else:
            archetype_scores["THE NIGHT OWL"][participant] = calculating_percentage(night, total)

In [14]:
calculating_night_owl_scores(valid_chat_data)

print("=" * 70)
print("THE NIGHT OWL")
print("=" * 70)

top_night_owls = sorted(
    archetype_scores["THE NIGHT OWL"].items(),
    key=lambda item: item[1],
    reverse=True
)

print(f"{'Rank':<6}{'Participant':<35}{'Night %'}")
print("-" * 70)

for rank, (participant, score) in enumerate(top_night_owls[:10], start=1):
    print(f"{rank:<6}{participant:<35}{score:.2f}")

THE NIGHT OWL
Rank  Participant                        Night %
----------------------------------------------------------------------
1     Ronil CSE                          41.75
2     Amisha CSE                         33.62
3     অর্ক Arka CSE HITK                 23.32
4     +91 82405 93621                    21.88
5     Anubhaw CSE SECTON 2               20.85
6     Debajyoti CSE                      19.75
7     Nirmalya CSE                       18.55
8     Mandrita Dasgupta  Jalpaiguri CSE  18.42
9     Ritesh CSE                         16.07
10    +91 96470 37567                    15.62


In [15]:
# THE STORYTELLER

def calculating_storyteller_scores(messages_per_person):
    for participant in participants:
        messages = messages_per_person[participant]
        total_messages = len(messages)

        if total_messages < MIN_MESSAGES:
            archetype_scores["THE STORYTELLER"][participant] = 0
            continue

        total_words = 0

        for message in messages:
            total_words += calculating_words(message["text"])

        archetype_scores["THE STORYTELLER"][participant] = total_words / total_messages

In [16]:
calculating_storyteller_scores(messages_per_person)

print("=" * 70)
print("THE STORYTELLER")
print("=" * 70)

top_storytellers = sorted(
    archetype_scores["THE STORYTELLER"].items(),
    key=lambda item: item[1],
    reverse=True
)

print(f"{'Rank':<6}{'Participant':<35}{'Avg Words'}")
print("-" * 70)

for rank, (participant, score) in enumerate(top_storytellers[:10], start=1):
    print(f"{rank:<6}{participant:<35}{score:.2f}")

THE STORYTELLER
Rank  Participant                        Avg Words
----------------------------------------------------------------------
1     +91 75469 52847                    8.63
2     +91 62910 01461                    7.52
3     Debayan Bhowmik CSE                6.69
4     +91 72788 80557                    6.27
5     +91 91993 26358                    6.21
6     +91 62915 39522                    6.06
7     Sayan Sheel CSE                    5.94
8     +91 80845 03900                    5.61
9     Souryadip (CSE)                    5.42
10    Anubhaw CSE SECTON 2               5.40


In [17]:
# THE DRAMA QUEEN

MIN_MESSAGES = 20

def calculating_drama_queen_scores(messages_per_person):
    for participant in participants:
        messages = messages_per_person[participant]
        total_messages = len(messages)

        if total_messages < MIN_MESSAGES:
            archetype_scores["THE DRAMA QUEEN"][participant] = 0
            continue

        drama_score = 0

        for message in messages:
            text = message["text"]

            drama_score += text.count("!")
            drama_score += text.count("?")

            words = extracting_words(text)
            drama_score += counting_keywords(words, DRAMA_KEYWORDS)

        archetype_scores["THE DRAMA QUEEN"][participant] = drama_score / total_messages

In [18]:
calculating_drama_queen_scores(messages_per_person)

print("=" * 70)
print("THE DRAMA QUEEN")
print("=" * 70)

top_drama_queens = sorted(
    archetype_scores["THE DRAMA QUEEN"].items(),
    key=lambda item: item[1],
    reverse=True
)

print(f"{'Rank':<6}{'Participant':<35}{'Drama Score'}")
print("-" * 70)

for rank, (participant, score) in enumerate(top_drama_queens[:10], start=1):
    print(f"{rank:<6}{participant:<35}{score:.2f}")

THE DRAMA QUEEN
Rank  Participant                        Drama Score
----------------------------------------------------------------------
1     Debeshee Sen CSE                   1.08
2     Ronil CSE                          1.01
3     Amisha CSE                         0.97
4     Debayan Bhowmik CSE                0.52
5     Soumili Ghosh CSE                  0.51
6     Sayan Sheel CSE                    0.48
7     +91 73842 52736                    0.41
8     Karan CSE                          0.39
9     Souryadip (CSE)                    0.37
10    Shubhradeep CSE                    0.36


In [19]:
# Only for analysis

print(type(response_analysis))
print(response_analysis)

<class 'dict'>
{'average_response_time': {'Anshley CSE A': 26.75056642636457, 'Kaninika CSE': 17.033120874117916, '+91 72788 80557': 17.243932038834952, 'Soumili Ghosh CSE': 36.71099958176495, 'Shubham Biswas': 28.29821860567407, 'Sayan Sheel CSE': 25.06756756756757, 'Shambaditya Sarkar CSE': 24.245363010068893, '+91 73842 52736': 4.265753424657534, '+91 70017 73199': 30.156288156288156, 'Amit CSE': 15.816793893129772, 'Nikhil CSE': 4.6008968609865475, 'Sohom Cse': 3.3333333333333335, 'Saptarshi CSE': 17.670811130846655, 'Sagnik CSE': 18.53117303953338, 'Shubham CSE': 9.353383458646617, 'Souryadip (CSE)': 15.679738562091503, 'Mohok CSE': 19.426716604244692, 'Kausik CSE A1': 12.499130434782609, 'Souryadip Cse': 2.7151736745886654, '𝐒𝐚𝐦 CSE': 95.09411764705882, '+91 95936 50398': 35.22141997593261, 'Ayush Kr Modi CSE': 41.925428784489185, 'Aritra Cse': 6.816275167785235, 'অর্ক Arka CSE HITK': 55.61077844311377, 'Shubham CSE Lambu': 51.42306279838414, 'Nirmalya CSE': 10.416666666666666, '

In [20]:
def calculating_ghost_scores(response_analysis, messages_per_person):
    silent_streaks = response_analysis["silent_streaks"]

    for participant in participants:

        message_count = len(messages_per_person[participant])

        if message_count < MIN_MESSAGES:
            archetype_scores["THE GHOST"][participant] = 0
            continue

        if participant in silent_streaks:
            silent_days = silent_streaks[participant]["days"]
        else:
            silent_days = 0

        ghost_score = silent_days / message_count

        archetype_scores["THE GHOST"][participant] = ghost_score

In [21]:
calculating_ghost_scores(response_analysis, messages_per_person)

print("=" * 70)
print("THE GHOST")
print("=" * 70)

top_ghosts = sorted(
    archetype_scores["THE GHOST"].items(),
    key=lambda item: item[1],
    reverse=True
)

print(f"{'Rank':<6}{'Participant':<35}{'Ghost Score'}")
print("-" * 70)

for rank, (participant, score) in enumerate(top_ghosts[:10], start=1):
    print(f"{rank:<6}{participant:<35}{score:.2f}")

THE GHOST
Rank  Participant                        Ghost Score
----------------------------------------------------------------------
1     Karan CSE                          32.89
2     +91 80171 72495                    23.49
3     +91 96415 12226                    18.25
4     +91 96470 37567                    13.53
5     Sayan Sheel CSE                    12.23
6     Ujjwal Kumar Mehta CSE             8.38
7     𝐒𝐚𝐦 CSE                            7.34
8     +91 82405 93621                    6.12
9     Ritesh CSE                         5.68
10    +91 80845 03900                    5.18


In [22]:
# THE COMEDIAN

def calculating_comedy_messages(participant_messages):
    """
    Counts the number of comedy messages sent by a participant.
    """

    comedy_messages = 0

    for message in participant_messages:
        text = message["text"]
        words = extracting_words(text)

        keyword_count = counting_keywords(words, COMEDIAN_KEYWORDS)

        contains_emoji = False
        for emoji in COMEDIAN_EMOJIS:
            if emoji in text:
                contains_emoji = True
                break

        if keyword_count > 0 or contains_emoji:
            comedy_messages += 1

    return comedy_messages

In [23]:
def calculating_comedian_scores(messages_per_person):
    """
    Calculates the Comedian score for each participant.

    Comedy Score =
        Comedy Messages / Total Messages
    """

    for participant in participants:
        participant_messages = messages_per_person.get(participant, [])
        total_messages = len(participant_messages)

        if total_messages < MIN_MESSAGES:
            archetype_scores["THE COMEDIAN"][participant] = 0
            continue

        comedy_messages = calculating_comedy_messages(participant_messages)

        archetype_scores["THE COMEDIAN"][participant] = comedy_messages / total_messages

calculating_comedian_scores(messages_per_person)

In [24]:
print("=" * 70)
print("THE COMEDIAN")
print("=" * 70)
print(f"{'Rank':<6}{'Participant':<35}{'Comedy Score'}")
print("-" * 70)

ranking = sorted(
    archetype_scores["THE COMEDIAN"].items(),
    key=lambda x: x[1],
    reverse=True
)

for rank, (participant, score) in enumerate(ranking[:10], start=1):
    print(f"{rank:<6}{participant:<35}{score:.2f}")

THE COMEDIAN
Rank  Participant                        Comedy Score
----------------------------------------------------------------------
1     +91 91993 26358                    0.23
2     Nirmalya CSE                       0.15
3     Nikhil CSE                         0.15
4     Rupankar CSE Phy Lab               0.13
5     +91 96749 56166                    0.13
6     Souryadip (CSE)                    0.13
7     SUMAN DEB KUNDU                    0.07
8     Harsh CSE                          0.07
9     Akash Cse                          0.07
10    Souryadip Cse                      0.07


In [25]:
# THE QUESTION MASTER

def calculating_question_messages(participant_messages):
    """
    Counts the number of question messages sent by a participant.
    """

    question_messages = 0

    for message in participant_messages:
        text = message["text"].strip()

        if text.endswith("?"):
            question_messages += 1
            continue

        words = text.lower().split()

        if len(words) > 0:
            first_word = words[0].strip(".,!?;:\"'()[]{}")

            if first_word in QUESTION_KEYWORDS:
                question_messages += 1

    return question_messages

In [26]:
def calculating_question_master_scores(messages_per_person):
    """
    Calculates the Question Master score for each participant.

    Question Score =
        Question Messages / Total Messages
    """

    for participant in participants:
        participant_messages = messages_per_person.get(participant, [])
        total_messages = len(participant_messages)

        if total_messages < MIN_MESSAGES:
            archetype_scores["THE QUESTION MASTER"][participant] = 0
            continue

        question_messages = calculating_question_messages(participant_messages)

        archetype_scores["THE QUESTION MASTER"][participant] = question_messages / total_messages

calculating_question_master_scores(messages_per_person)

In [27]:
print("=" * 70)
print("THE QUESTION MASTER")
print("=" * 70)
print(f"{'Rank':<6}{'Participant':<35}{'Question Score'}")
print("-" * 70)

ranking = sorted(
    archetype_scores["THE QUESTION MASTER"].items(),
    key=lambda x: x[1],
    reverse=True
)

for rank, (participant, score) in enumerate(ranking[:10], start=1):
    print(f"{rank:<6}{participant:<35}{score:.2f}")

THE QUESTION MASTER
Rank  Participant                        Question Score
----------------------------------------------------------------------
1     Debayan Bhowmik CSE                0.18
2     +91 73842 52736                    0.17
3     +91 74781 83062                    0.17
4     Soumili Ghosh CSE                  0.16
5     Ujjwal Kumar Mehta CSE             0.14
6     +91 62910 01461                    0.14
7     Sayan Sheel CSE                    0.14
8     𝐒𝐚𝐦 CSE                            0.14
9     +91 72788 80557                    0.14
10    Shubhradeep CSE                    0.13


In [28]:
def normalizing_archetype_scores():
    """
    Normalizes each archetype score to the range [0, 1]
    using Min-Max Normalization.
    """

    normalized_scores = {}

    for archetype in archetype_scores:

        normalized_scores[archetype] = {}

        scores = list(archetype_scores[archetype].values())

        minimum_score = min(scores)
        maximum_score = max(scores)

        for participant, score in archetype_scores[archetype].items():

            if maximum_score == minimum_score:
                normalized_scores[archetype][participant] = 0

            else:
                normalized_scores[archetype][participant] = (
                    (score - minimum_score) /
                    (maximum_score - minimum_score)
                )

    return normalized_scores


normalized_archetype_scores = normalizing_archetype_scores()

In [29]:
# FINAL PERSONALITY ANALYSIS

def assigning_personality_archetypes():
    """
    Assigns the dominant personality archetype to each participant
    based on the highest archetype score.
    """

    personality_archetypes = {}

    for participant in participants:

        best_archetype = None
        best_score = -1

        for archetype in normalized_archetype_scores:
            score = normalized_archetype_scores[archetype].get(participant, 0)

            if score > best_score:
                best_score = score
                best_archetype = archetype

        personality_archetypes[participant] = {
            "Archetype": best_archetype,
            "Score": best_score
        }

    return personality_archetypes


In [30]:
personality_archetypes = assigning_personality_archetypes()

print("=" * 90)
print("FINAL PERSONALITY ASSIGNMENT")
print("=" * 90)
print(f"{'Participant':<35}{'Archetype':<30}{'Score'}")
print("-" * 90)

for participant in sorted(personality_archetypes):

    archetype = personality_archetypes[participant]["Archetype"]
    score = personality_archetypes[participant]["Score"]

    print(f"{participant:<35}{archetype:<30}{score:.2f}")

FINAL PERSONALITY ASSIGNMENT
Participant                        Archetype                     Score
------------------------------------------------------------------------------------------
+91 62910 01461                    THE STORYTELLER               0.87
+91 62915 39522                    THE STORYTELLER               0.70
+91 70017 73199                    THE SPAMMER                   0.66
+91 72788 80557                    THE QUESTION MASTER           0.75
+91 73842 52736                    THE QUESTION MASTER           0.93
+91 74781 83062                    THE QUESTION MASTER           0.92
+91 74881 27294                    THE SPAMMER                   0.58
+91 75469 52847                    THE STORYTELLER               1.00
+91 79084 39726                    THE GROUP MOM                 0.00
+91 79804 47155                    THE SPAMMER                   0.19
+91 80171 72495                    THE GHOST                     0.71
+91 80845 03900                    THE 

In [31]:
print("=" * 70)
print("VALIDATION")
print("=" * 70)

print(f"Total Participants          : {len(participants)}")
print(f"Assigned Participants       : {len(personality_archetypes)}")

missing = set(participants) - set(personality_archetypes.keys())
print(f"Missing Assignments         : {len(missing)}")

if len(missing) == 0:
    print("Assignment Status           : PASSED")
else:
    print("Assignment Status           : FAILED")

minimum_score = 1
maximum_score = 0

for archetype in normalized_archetype_scores:
    scores = normalized_archetype_scores[archetype].values()

    minimum_score = min(minimum_score, min(scores))
    maximum_score = max(maximum_score, max(scores))

print(f"Normalized Score Range      : {minimum_score:.2f} - {maximum_score:.2f}")

print("\nParticipants per Archetype")
print("-" * 70)

archetype_counts = {}

for details in personality_archetypes.values():
    archetype = details["Archetype"]
    archetype_counts[archetype] = archetype_counts.get(archetype, 0) + 1

for archetype in sorted(archetype_counts):
    print(f"{archetype:<30}: {archetype_counts[archetype]}")

VALIDATION
Total Participants          : 75
Assigned Participants       : 75
Missing Assignments         : 0
Assignment Status           : PASSED
Normalized Score Range      : 0.00 - 1.00

Participants per Archetype
----------------------------------------------------------------------
THE COMEDIAN                  : 4
THE DRAMA QUEEN               : 2
THE GHOST                     : 3
THE GROUP MOM                 : 3
THE NIGHT OWL                 : 4
THE QUESTION MASTER           : 11
THE SPAMMER                   : 22
THE STORYTELLER               : 26


In [32]:
feature7_summary = {
    "normalized_scores": normalized_archetype_scores,
    "assignments": personality_archetypes
}

with open(DATA_PATH + "personality_archetypes.pkl", "wb") as file:
    pickle.dump(feature7_summary, file)

print("personality_archetypes.pkl exported successfully.")

personality_archetypes.pkl exported successfully.
